# 02 · Training Loop, Optimizers and LR Schedules

Write the PyTorch training loop from memory, then learn the knobs that decide whether it trains at all: the
optimizer, the learning rate and its schedule, gradient clipping, batch size, early stopping, seeds and numeric
precision. Every claim is measured on a small handwritten-digits classifier, so you can quote results rather than
slogans.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · a neuron and backprop](01_neuron_and_backprop_from_scratch.ipynb); [PyTorch essentials](../04_numpy_pandas_pytorch/03_pytorch_essentials.ipynb) for tensors, `nn.Module` and autograd

## Why this matters in interviews

- "Write a training loop" is a live-coding staple. They check the order (`zero_grad` → forward → loss → `backward`
  → clip → `step` → schedule) and that you switch to `eval()` + `no_grad()` for validation.
- The LLM recipe is standard: AdamW, linear warmup then cosine decay, gradient clipping at 1.0, bf16 autocast,
  gradient accumulation to reach a large batch. You should be able to justify each piece.
- Debugging questions ("the loss is NaN", "the loss plateaus", "val loss goes up") are answered by exactly the plots
  in this notebook.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn22` | What is the difference between an epoch, a step and a batch in LLM training? |
| `tn36` | What is mixed-precision training and what is bf16 for? |
| `tn35` | What is gradient checkpointing and why would you use it? |

In [ ]:
import copy
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

random.seed(0); np.random.seed(0); torch.manual_seed(0)
torch.set_num_threads(1)                         # tiny models: thread start-up costs more than the maths
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.8})

X, y = load_digits(return_X_y=True)             # 1,797 images of 8x8 pixels, digits 0-9
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
scaler = StandardScaler().fit(X_tr)             # fit on train only, or validation leaks into training
X_tr = torch.tensor(scaler.transform(X_tr), dtype=torch.float32)
X_va = torch.tensor(scaler.transform(X_va), dtype=torch.float32)
y_tr, y_va = torch.tensor(y_tr), torch.tensor(y_va)

def make_model(seed=0, hidden=64):
    torch.manual_seed(seed)                     # same seed -> same initial weights
    return nn.Sequential(nn.Linear(64, hidden), nn.ReLU(), nn.Linear(hidden, 10))

print(f"train {tuple(X_tr.shape)}  val {tuple(X_va.shape)}")
print(make_model(), f"\nparameters: {sum(p.numel() for p in make_model().parameters()):,}")

## 1. The training loop

**In plain English:** show the model a small batch, measure how wrong it is, work out which way to nudge every
weight, nudge them a little, repeat. After every full pass over the data, check the model on data it never trains on.

The vocabulary, used precisely:

| Term | Meaning | Here |
|---|---|---|
| batch | the examples in one forward/backward pass (LLM batches are counted in *tokens*) | 64 images |
| step | one optimizer update, i.e. one batch | |
| epoch | one full pass over the training set | ⌈1,437 / 64⌉ = 23 steps |

```text
for epoch in range(epochs):
    model.train()                        # dropout / BatchNorm in training mode
    for xb, yb in batches:               # reshuffled every epoch
        optimizer.zero_grad()            # 1. gradients accumulate, so clear them
        loss = loss_fn(model(xb), yb)    # 2. forward pass + loss on raw logits
        loss.backward()                  # 3. gradient for every parameter
        clip_grad_norm_(params, 1.0)     #    optional: bound the update size
        optimizer.step()                 # 4. update the weights
        scheduler.step()                 # 5. optional: per-step learning-rate schedule
    model.eval(); with torch.no_grad(): validate
```

In [ ]:
def batches(X, Y, batch_size, generator):
    """Reshuffle, then yield mini-batches: DataLoader(shuffle=True) does the same for in-memory tensors."""
    perm = torch.randperm(len(X), generator=generator)
    for i in range(0, len(X), batch_size):
        idx = perm[i:i + batch_size]
        yield X[idx], Y[idx]

def evaluate(model, X, Y):
    model.eval()                                   # dropout off, BatchNorm uses running statistics
    with torch.no_grad():                          # no autograd graph: less memory, faster
        logits = model(X)
    return F.cross_entropy(logits, Y).item(), (logits.argmax(1) == Y).float().mean().item()

def fit(model, opt, epochs, batch_size=64, scheduler=None, clip=None, X=X_tr, Y=y_tr, seed=0):
    gen = torch.Generator().manual_seed(seed)      # reproducible shuffling
    hist = {"train": [], "val": [], "step_loss": []}
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in batches(X, Y, batch_size, gen):
            opt.zero_grad()                        # 1
            loss = F.cross_entropy(model(xb), yb)  # 2
            loss.backward()                        # 3
            if clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()                             # 4
            if scheduler is not None:
                scheduler.step()                   # 5
            loss_value = loss.item()
            total += loss_value * len(xb)
            hist["step_loss"].append(loss_value)
        hist["train"].append(total / len(X))       # mean loss over the epoch, while the weights moved
        hist["val"].append(evaluate(model, X_va, y_va)[0])
    return hist

from torch.utils.data import DataLoader, TensorDataset
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(0))
xb, yb = next(iter(loader))
print(f"DataLoader: {len(loader)} batches per epoch, first batch {tuple(xb.shape)} and {tuple(yb.shape)}")
assert len(loader) == sum(1 for _ in batches(X_tr, y_tr, 64, torch.Generator())) == math.ceil(len(X_tr) / 64)

With files on disk, augmentation or multiple worker processes, use `DataLoader`. For a small tensor that already
sits in memory, `batches()` does the same job with less overhead. Now train with Adam:

In [ ]:
model = make_model()
t0 = time.time()
hist = fit(model, torch.optim.Adam(model.parameters(), lr=1e-3), epochs=20)
val_loss, val_acc = evaluate(model, X_va, y_va)
print(f"20 epochs x 23 steps in {time.time() - t0:.1f}s   val loss {val_loss:.3f}   val accuracy {val_acc:.3f}")
assert val_acc > 0.95

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, 21), hist["train"], label="train (mean during the epoch)")
ax.plot(range(1, 21), hist["val"], label="validation (end of epoch)")
ax.set(title="Train vs validation loss, Adam lr=1e-3", xlabel="epoch", ylabel="cross-entropy", yscale="log")
ax.legend(); plt.show()

Both curves fall together at first. The widening gap later on is the model fitting training-set detail that does not
carry over; [03 · regularization](03_regularization_dropout_batchnorm.ipynb) is about controlling that gap. (Early
on, train can sit *above* val: it is averaged over an epoch in which the weights were still improving.)

## 2. Optimizers compared on the same problem

| Optimizer | Update for each weight | Extra memory per weight | Typical use |
|---|---|---|---|
| SGD | `w -= lr * g` | none | baseline |
| SGD + momentum | `v = μ·v + g;  w -= lr * v` | 1 number (`v`) | CNN recipes (ResNet) |
| Adam | `w -= lr * m̂ / (√v̂ + ε)` | 2 numbers (`m`, `v`) | default for most deep nets |
| AdamW | Adam + *decoupled* weight decay `w -= lr·λ·w` | 2 numbers | transformers and LLMs |

Same initial weights, same batch order, 8 epochs each:

In [ ]:
OPTIMIZERS = {
    "SGD, lr 0.01": lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD + momentum 0.9, lr 0.01": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam, lr 1e-3": lambda p: torch.optim.Adam(p, lr=1e-3),
    "AdamW, lr 1e-3, wd 0.01": lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=0.01),
}
runs = {}
for name, make_opt in OPTIMIZERS.items():
    model = make_model()                                 # identical starting weights for every optimizer
    runs[name] = fit(model, make_opt(model.parameters()), epochs=8)
    runs[name]["acc"] = evaluate(model, X_va, y_va)[1]

fig, ax = plt.subplots(figsize=(10, 4))
for name, h in runs.items():
    ax.plot(range(1, 9), h["train"], marker="o", markersize=4, label=name)
ax.set(title="Training loss per epoch, same model, same data order", xlabel="epoch", ylabel="train loss (log)",
       yscale="log")
ax.legend(); plt.show()
for name, h in runs.items():
    print(f"{name:28s} final train loss {h['train'][-1]:.3f}   val accuracy {h['acc']:.3f}")
sgd_final = runs["SGD, lr 0.01"]["train"][-1]
assert runs["SGD + momentum 0.9, lr 0.01"]["train"][-1] < sgd_final / 3
assert runs["Adam, lr 1e-3"]["train"][-1] < sgd_final / 3

Plain SGD at lr 0.01 crawls. Momentum 0.9 keeps adding the previous step, which multiplies the effective step by
about 1/(1 − 0.9) = 10 along directions the gradient keeps pointing, so the same lr 0.01 goes much faster. Adam gets
there without tuning because it rescales every weight's step (next section). AdamW overlaps Adam here: it differs
only in weight decay, which is small over 8 epochs; its benefit is regularisation, not speed.

This is **not a fair race**: every optimizer has its own best learning rate (section 4). The honest summary is
"Adam/AdamW are robust defaults; tuned SGD + momentum is still competitive on vision".

## 3. What Adam's two moments do

$$m \leftarrow \beta_1 m + (1-\beta_1)\,g \qquad v \leftarrow \beta_2 v + (1-\beta_2)\,g^2$$
$$\hat m = \frac{m}{1-\beta_1^t},\quad \hat v = \frac{v}{1-\beta_2^t},\qquad w \leftarrow w - \mathrm{lr}\cdot\frac{\hat m}{\sqrt{\hat v}+\epsilon}$$

- `m` (first moment) is a running mean of the gradient: momentum, a smoothed *direction*.
- `v` (second moment) is a running mean of the squared gradient: each weight's own gradient *scale*. Dividing by
  `√v̂` makes the step size roughly independent of how large that weight's gradients are.
- Both start at 0, so early averages are biased towards 0; dividing by `1 − β^t` corrects that.

Check a hand-written Adam against `torch.optim.Adam`, feeding gradients whose sizes differ by a factor of a million:

In [ ]:
torch.manual_seed(0)
scales = torch.tensor([1e-3, 1e-1, 1.0, 10.0, 1e3])      # per-coordinate gradient sizes spanning 10^6
w = nn.Parameter(torch.zeros(5))
opt = torch.optim.Adam([w], lr=0.1, betas=(0.9, 0.999), eps=1e-8)
w_hand, m, v = torch.zeros(5), torch.zeros(5), torch.zeros(5)
b1, b2, lr, eps = 0.9, 0.999, 0.1, 1e-8
for t in range(1, 4):
    g = scales * (1 + 0.1 * torch.randn(5))              # noisy gradients, one scale per coordinate
    before = w.detach().clone()
    w.grad = g.clone()
    opt.step()
    m = b1 * m + (1 - b1) * g                            # 1st moment: direction
    v = b2 * v + (1 - b2) * g ** 2                       # 2nd moment: scale
    m_hat, v_hat = m / (1 - b1 ** t), v / (1 - b2 ** t)  # bias correction
    w_hand -= lr * m_hat / (v_hat.sqrt() + eps)
    assert torch.allclose(w.detach(), w_hand, atol=1e-6)
    if t == 1:
        step_size = (w.detach() - before).abs()
        print("gradient sizes    :", ["%.0e" % x for x in g.abs().tolist()])
        print("first-step |update|:", ["%.4f" % x for x in step_size.tolist()])
        assert torch.allclose(step_size, torch.full((5,), lr), rtol=1e-4)   # every weight moves by lr
print("hand-written Adam == torch.optim.Adam for 3 steps")

On step 1, `m̂ = g` and `v̂ = g²`, so the update is `lr · g/|g|`: every weight moves by exactly `lr`, whether its
gradient is 0.001 or 1000. That scale-invariance is why Adam needs little tuning across layers, and also why its
learning rate means "roughly the maximum step per weight". The price is memory: two extra numbers per parameter.

## 4. Learning-rate sweep

The learning rate is the hyperparameter that matters most. Sweep it on a log grid (~3× apart) and train each for
just one epoch (23 steps):

In [ ]:
lrs = np.logspace(-4, 1, 11)                             # 1e-4 ... 10
SWEEP = {"SGD + momentum 0.9": lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9),
         "Adam": lambda p, lr: torch.optim.Adam(p, lr=lr)}
sweep = {}
for name, make_opt in SWEEP.items():
    losses = []
    for lr in lrs:
        model = make_model()
        fit(model, make_opt(model.parameters(), lr), epochs=1)
        losses.append(evaluate(model, X_tr, y_tr)[0])     # where the weights ended up
    sweep[name] = np.array(losses)

init_loss, CAP = evaluate(make_model(), X_tr, y_tr)[0], 30.0   # the untrained model's loss (about ln 10)
fig, ax = plt.subplots(figsize=(10, 4))
for name, losses in sweep.items():
    shown = np.where(np.isfinite(losses), np.minimum(losses, CAP), CAP)
    line, = ax.plot(lrs, shown, marker="o", markersize=5, label=name)
    bad = shown > init_loss
    ax.scatter(lrs[bad], shown[bad], marker="x", s=90, color=line.get_color(), zorder=3)
ax.axhline(init_loss, color="#898781", lw=1)
ax.text(1.2e-4, init_loss * 1.25, "loss at init: points above it (x) got worse than untrained, or diverged", fontsize=8)
ax.set(xscale="log", yscale="log", title="Learning-rate sweep: training loss after 1 epoch (23 steps)",
       xlabel="learning rate (log scale)", ylabel=f"train loss (log, NaN shown at {CAP:.0f})")
ax.legend(); plt.show()

print(f"{'lr':>8s} " + " ".join(f"{n:>20s}" for n in sweep))
for i, lr in enumerate(lrs):
    print(f"{lr:8.0e} " + " ".join(f"{sweep[n][i]:20.4g}" for n in sweep))
sgd, adam = sweep["SGD + momentum 0.9"], sweep["Adam"]
assert sgd[0] > 2.0 and adam[0] > 2.0                                  # too low: barely moved
assert not np.isfinite(sgd[-1]) or sgd[-1] > init_loss                 # too high: diverged
assert 1e-2 <= lrs[np.nanargmin(sgd)] <= 0.3 and 1e-3 <= lrs[np.nanargmin(adam)] <= 0.05

The classic valley. Too small and nothing happens in the time available; too large and the loss explodes, ending
in `nan` (the most common cause of a NaN loss). Adam's good range sits about 10× lower than SGD + momentum's here,
because its steps are already normalised. In practice: pick a value a little below the bottom of the valley, since
the edge of stability is not where you want to live for a long run.

## 5. Learning-rate schedules

A schedule changes the learning rate over training. `torch.optim.lr_scheduler` has them all; call
`scheduler.step()` once per optimizer step, *after* `optimizer.step()`. Plot four, as multipliers on a peak LR of 1:

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, ExponentialLR, LambdaLR, LinearLR, SequentialLR, StepLR
TOTAL, WARMUP = 1000, 100

def lr_curve(make_scheduler, steps=TOTAL):
    """Drive a scheduler on a dummy optimizer (base LR 1.0) and record the LR used at every step."""
    opt = torch.optim.SGD([nn.Parameter(torch.zeros(1))], lr=1.0)
    sched, lrs_seen = make_scheduler(opt), []
    for _ in range(steps):
        lrs_seen.append(opt.param_groups[0]["lr"])
        opt.step()
        sched.step()
    return np.array(lrs_seen)

def warmup_cosine(step, warmup=WARMUP, total=TOTAL, floor=0.1):
    """Linear warmup to the peak, then cosine decay to 10% of the peak."""
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / (total - warmup)
    return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * progress))

schedules = {
    "step: x0.5 every 250 steps": lr_curve(lambda o: StepLR(o, step_size=250, gamma=0.5)),
    "exponential: x0.997 per step": lr_curve(lambda o: ExponentialLR(o, gamma=0.997)),
    "cosine": lr_curve(lambda o: CosineAnnealingLR(o, T_max=TOTAL)),
    "linear warmup + cosine (LLM standard)": lr_curve(lambda o: LambdaLR(o, warmup_cosine)),
}
fig, ax = plt.subplots(figsize=(10, 4))
for name, curve in schedules.items():
    ax.plot(curve, label=name)
ax.set(title="Learning-rate schedules", xlabel="optimizer step", ylabel="LR / peak LR")
ax.legend(); plt.show()

wc = schedules["linear warmup + cosine (LLM standard)"]
builtin = lr_curve(lambda o: SequentialLR(o, [LinearLR(o, start_factor=1 / WARMUP, total_iters=WARMUP - 1),
                                              CosineAnnealingLR(o, T_max=TOTAL - WARMUP, eta_min=0.1)],
                                          milestones=[WARMUP]))
assert np.allclose(builtin, wc)                          # the same schedule from built-in pieces
assert math.isclose(wc[0], 1 / WARMUP) and math.isclose(wc[WARMUP - 1], 1.0) and abs(wc[-1] - 0.1) < 1e-3
assert schedules["step: x0.5 every 250 steps"][250] == 0.5 and schedules["cosine"][-1] < 1e-4
print(f"warmup+cosine: starts at {wc[0]:.2f}, peaks at step {int(np.argmax(wc))}, ends at {wc[-1]:.3f}")

**Why warm up?** At step 1 the weights are random and Adam's `v` is estimated from a single gradient, so full-size
steps can throw the model somewhere it never recovers from (post-norm transformers are notorious for this). A few
hundred to a few thousand warmup steps let the statistics settle first.

**Why decay?** A large LR makes fast progress but bounces around the minimum; shrinking it lets the model settle
into the bottom of the valley. Cosine does this smoothly with one knob (the length). LLM pretraining recipes (GPT-3,
Llama) use linear warmup, then cosine decay to about 10% of the peak. In Hugging Face `Trainer` it is
`lr_scheduler_type="cosine"` with `warmup_steps` or `warmup_ratio`.

## 6. Gradient clipping

`clip_grad_norm_` treats *all* gradients as one long vector; if its length exceeds `max_norm`, it rescales the
whole vector down to `max_norm`. The direction is unchanged, only the step size is capped. It returns the norm
*before* clipping, which is the number to log.

In [ ]:
params = [nn.Parameter(torch.zeros(3)), nn.Parameter(torch.zeros(2))]
params[0].grad, params[1].grad = torch.tensor([3.0, 4.0, 0.0]), torch.tensor([0.0, 12.0])
before = torch.cat([p.grad for p in params]).clone()
total_norm = torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
after = torch.cat([p.grad for p in params])
print(f"global norm before: {total_norm.item():.1f}   after: {after.norm().item():.4f}")
print("same direction:", torch.allclose(after, before / before.norm(), atol=1e-6))
assert math.isclose(total_norm.item(), 13.0) and torch.allclose(after, before / 13, atol=1e-6)

Why it matters: one bad batch (a corrupted record, an outlier, a rare token) can produce a gradient hundreds of
times larger than usual, and a single huge step can wreck a run. Simulate it with three corrupted batches (inputs
scaled by 50, labels scrambled), and train twice (SGD + momentum, lr 0.1), once without clipping and once with `max_norm=1.0`:

In [ ]:
BAD_STEPS, SCALE = (40, 90, 140), 50.0

def run_with_spikes(clip, steps=200, lr=0.1):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    gen, norms, losses, step = torch.Generator().manual_seed(0), [], [], 0
    while step < steps:
        for xb, yb in batches(X_tr, y_tr, 64, gen):
            if step in BAD_STEPS:                         # a corrupted batch: huge inputs, garbage labels
                xb = xb * SCALE
                yb = torch.randint(0, 10, yb.shape, generator=torch.Generator().manual_seed(step))
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)   # clip=inf only measures
            opt.step()
            norms.append(norm.item()); losses.append(loss.item()); step += 1
            if step == steps:
                break
    return np.array(norms), np.array(losses), evaluate(model, X_va, y_va)[0]

no_clip, clipped = run_with_spikes(clip=float("inf")), run_with_spikes(clip=1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for (norms, losses, _), name in [(no_clip, "no clipping"), (clipped, "clip_grad_norm_(max_norm=1.0)")]:
    ax1.plot(norms, label=name); ax2.plot(losses, label=name)
for ax in (ax1, ax2):
    for s in BAD_STEPS:
        ax.axvline(s, color="#c3c2b7", lw=1)
    ax.set(yscale="log", xlabel="step (grey lines: corrupted batches)")
ax1.axhline(1.0, color="#52514e", lw=1)
ax1.set(title="Gradient norm before clipping", ylabel="global grad norm (log)")
ax2.set(title="Training loss", ylabel="cross-entropy (log)")
ax1.legend(); plt.tight_layout(); plt.show()
print(f"final val loss   no clipping: {no_clip[2]:.3f}   with clipping: {clipped[2]:.3f}")
assert no_clip[2] > 5 * clipped[2]

Both runs see the same spikes in the raw gradient norm (left). Without clipping, a spike turns into a giant step, the
loss explodes and the run never recovers. With clipping, the spike still happens but the step is capped, so the loss
barely notices. That is why LLM training clips at 1.0 by default and why a *rising* gradient-norm curve is an early
warning to watch on a dashboard.

## 7. Batch size

**In plain English:** the gradient from a batch is an estimate of the gradient over the whole training set. More
examples means a less noisy estimate, but the noise shrinks only with the *square root* of the batch size. Measure
it: draw many batches of each size B, and compare each batch gradient with the full-dataset gradient.

In [ ]:
probe = make_model()

def flat_grad(xb, yb):
    probe.zero_grad()
    F.cross_entropy(probe(xb), yb).backward()
    return torch.cat([p.grad.flatten() for p in probe.parameters()]).clone()

g_full = flat_grad(X_tr, y_tr)
gen = torch.Generator().manual_seed(0)
Bs = [4, 8, 16, 32, 64, 128, 256, 512]
noise = []
for B in Bs:
    errs = [(flat_grad(X_tr[idx], y_tr[idx]) - g_full).norm().item()
            for idx in (torch.randint(len(X_tr), (B,), generator=gen) for _ in range(30))]
    noise.append(np.mean(errs))
slope = np.polyfit(np.log(Bs), np.log(noise), 1)[0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.loglog(Bs, noise, marker="o", label="measured: mean ||batch grad - full grad||")
ax.loglog(Bs, noise[0] * np.sqrt(Bs[0] / np.array(Bs)), color="#898781", lw=1, label="1/sqrt(B) reference")
ax.set(title=f"Gradient noise vs batch size (fitted slope {slope:.2f})", xlabel="batch size B", ylabel="gradient error")
ax.legend(); plt.show()
print(f"4x the batch -> {np.mean(np.array(noise[:-2]) / np.array(noise[2:])):.2f}x less noise (theory: 2x)")
assert -0.6 < slope < -0.4

So a 4× bigger batch buys a 2× cleaner gradient, at 4× the compute per step. Now train with different batch sizes
for the same 6 epochs (the same number of examples seen):

In [ ]:
bs_runs = {}
for B, lr in [(16, 1e-3), (64, 1e-3), (256, 1e-3), (256, 1.6e-2)]:
    model = make_model()
    t0 = time.time()
    h = fit(model, torch.optim.Adam(model.parameters(), lr=lr), epochs=6, batch_size=B)
    bs_runs[f"B={B}, lr={lr:g}"] = dict(h, steps=6 * math.ceil(len(X_tr) / B), secs=time.time() - t0)

fig, ax = plt.subplots(figsize=(10, 4))
for name, h in bs_runs.items():
    ax.plot(range(1, 7), h["val"], marker="o", markersize=4, label=name)
ax.set(title="Validation loss for different batch sizes, same 6 epochs", xlabel="epoch", ylabel="val loss (log)",
       yscale="log")
ax.legend(); plt.show()
for name, h in bs_runs.items():
    print(f"{name:18s} {h['steps']:4d} optimizer steps   final val loss {h['val'][-1]:.3f}   {h['secs']:.2f}s")
v = {k: h["val"][-1] for k, h in bs_runs.items()}
assert v["B=16, lr=0.001"] < v["B=64, lr=0.001"] < v["B=256, lr=0.001"]
assert v["B=256, lr=0.016"] < v["B=256, lr=0.001"] / 2

At a fixed learning rate and a fixed number of epochs, **small batches win**: B=16 takes 16× more optimizer steps
than B=256. Big batches are not worse, they need a bigger learning rate: the **linear scaling rule** (batch ×k →
LR ×k, usually with warmup) brings B=256 at lr 0.016 back close to the small-batch result in 16× fewer steps. On a
GPU, fewer, bigger steps is the whole point: the hardware processes a batch of 256 almost as fast as a batch of 16.
The rule breaks down past a "critical batch size", where extra examples stop reducing useful noise.

### Gradient accumulation: a big batch on a small GPU

If a batch of 64 does not fit in memory, run 4 micro-batches of 16, call `backward()` on each (gradients add up),
and step once. Divide each micro-batch loss by 4 so the sum equals the mean over 64:

In [ ]:
model = make_model()
xb, yb = X_tr[:64], y_tr[:64]
model.zero_grad()
F.cross_entropy(model(xb), yb).backward()
big_batch = [p.grad.clone() for p in model.parameters()]

ACCUM = 4
model.zero_grad()
for chunk_x, chunk_y in zip(xb.chunk(ACCUM), yb.chunk(ACCUM)):          # 4 micro-batches of 16
    (F.cross_entropy(model(chunk_x), chunk_y) / ACCUM).backward()       # scale so the sum is a mean
diff = max((a - p.grad).abs().max().item() for a, p in zip(big_batch, model.parameters()))
print(f"max |one batch of 64 - 4 accumulated micro-batches of 16| = {diff:.1e}")
assert diff < 1e-6
print(f"effective batch = micro-batch x accumulation steps x GPUs, e.g. 16 x {ACCUM} x 8 = {16 * ACCUM * 8}")

Two traps: BatchNorm sees only the micro-batch, so accumulation is *not* exactly equivalent with BatchNorm; and for
token-level LLM losses, normalise by the total number of tokens across all micro-batches, not by averaging per-micro-
batch means (micro-batches with different token counts would otherwise be weighted wrongly).

## 8. Early stopping on validation loss

Train loss almost always keeps falling. Validation loss falls, bottoms out, then rises as the model starts
memorising. Early stopping watches validation loss, remembers the best weights, and stops after `patience` epochs
without improvement. To make memorisation happen fast, train a bigger net on 300 examples with 30% of the labels
scrambled:

In [ ]:
class EarlyStopping:
    """Stop when val loss has not improved for `patience` epochs; keep a copy of the best weights."""
    def __init__(self, patience=10):
        self.patience, self.best, self.best_epoch, self.bad_epochs, self.best_state = patience, math.inf, -1, 0, None

    def should_stop(self, val_loss, model, epoch):
        if val_loss < self.best:
            self.best, self.best_epoch, self.bad_epochs = val_loss, epoch, 0
            self.best_state = copy.deepcopy(model.state_dict())      # a copy, not a reference
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

gen = torch.Generator().manual_seed(1)
X_small, y_small = X_tr[:300], y_tr[:300].clone()
flip = torch.rand(300, generator=gen) < 0.3
y_small[flip] = torch.randint(0, 10, (int(flip.sum()),), generator=gen)   # noisy labels

model = make_model(hidden=128)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
stopper, stop_epoch, tr_curve, va_curve = EarlyStopping(patience=10), None, [], []
shuffle = torch.Generator().manual_seed(0)
for epoch in range(60):
    model.train()
    for xb, yb in batches(X_small, y_small, 32, shuffle):
        opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    tr_curve.append(evaluate(model, X_small, y_small)[0]); va_curve.append(evaluate(model, X_va, y_va)[0])
    if stop_epoch is None and stopper.should_stop(va_curve[-1], model, epoch):
        stop_epoch = epoch        # a real loop would `break` here; we keep going to see what we avoided

In [ ]:
final_val, final_acc = evaluate(model, X_va, y_va)
model.load_state_dict(stopper.best_state)                 # restore the best checkpoint
best_val, best_acc = evaluate(model, X_va, y_va)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tr_curve, label="train loss (noisy labels)")
ax.plot(va_curve, label="validation loss")
ax.axvline(stopper.best_epoch, color="#1baf7a", lw=1.5, label=f"best epoch {stopper.best_epoch}")
ax.axvline(stop_epoch, color="#52514e", lw=1, label=f"early stop at epoch {stop_epoch} (patience 10)")
ax.set(title="Early stopping: stop when validation loss stops improving", xlabel="epoch", ylabel="cross-entropy",
       yscale="log")
ax.legend(); plt.show()
print(f"best epoch {stopper.best_epoch}: val loss {best_val:.3f}, accuracy {best_acc:.3f}")
print(f"epoch 59 without stopping: val loss {final_val:.3f}, accuracy {final_acc:.3f}")
assert math.isclose(best_val, stopper.best, rel_tol=1e-6)  # the restored weights are the best ones
assert final_val > 1.3 * best_val and best_acc > final_acc

The model memorises the scrambled labels (train loss towards 0) while validation loss climbs back up, and
validation accuracy falls with it. Early stopping kept the weights from the bottom of the valley. Two details
interviewers probe: restore the best weights (stopping alone leaves you with the last, worse ones), and pick
patience large enough that normal epoch-to-epoch noise does not trigger it.

## 9. Reproducibility

Seed every random number generator you use, and pass a seeded generator to anything that shuffles. With
`torch.use_deterministic_algorithms(True)`, PyTorch raises an error on any operation that has no deterministic
implementation instead of silently varying.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)   # torch.manual_seed also seeds CUDA

def train_once(seed):
    set_seed(seed)
    model = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))    # init draws from the global RNG
    fit(model, torch.optim.Adam(model.parameters(), lr=1e-3), epochs=2, seed=seed)
    return torch.cat([p.detach().flatten() for p in model.parameters()])

torch.use_deterministic_algorithms(True)
run_a, run_b, run_c = train_once(0), train_once(0), train_once(1)
torch.use_deterministic_algorithms(False)
print("seed 0 twice -> bit-identical weights:", torch.equal(run_a, run_b))
print("seed 0 vs 1  -> identical:            ", torch.equal(run_a, run_c))
assert torch.equal(run_a, run_b) and not torch.equal(run_a, run_c)

big, one = torch.tensor(1e8), torch.tensor(1.0)           # float32
print("float32: (1e8 + 1) - 1e8 =", ((big + one) - big).item(), "  but (1e8 - 1e8) + 1 =", ((big - big) + one).item())
assert ((big + one) - big).item() != ((big - big) + one).item()

The last line is why seeds are not enough on a GPU: floating-point addition is **not associative**, and parallel
hardware may sum in a different order from run to run (or depending on what else is in the batch). Checklist:

| Source of randomness | Fix |
|---|---|
| Python / NumPy / torch RNGs | `random.seed`, `np.random.seed`, `torch.manual_seed` |
| data shuffling | `DataLoader(..., generator=torch.Generator().manual_seed(s))`, plus `worker_init_fn` for workers |
| non-deterministic GPU kernels | `torch.use_deterministic_algorithms(True)`, `torch.backends.cudnn.benchmark = False`, env `CUBLAS_WORKSPACE_CONFIG=:4096:8` |
| reduction order, library versions, hardware | cannot be seeded: pin versions and compare with tolerances, not equality |

## 10. Mixed precision: fp32 vs fp16 vs bf16

**In plain English:** do most of the arithmetic with shorter numbers (16 bits instead of 32) so it is faster and
uses less memory, but keep the numbers that must stay accurate (the master weights, the optimizer state, the loss)
in 32 bits. The layout of each format explains everything:

In [ ]:
LAYOUT = {torch.float32: (1, 8, 23), torch.float16: (1, 5, 10), torch.bfloat16: (1, 8, 7)}   # sign/exponent/mantissa
print(f"{'dtype':9s} {'sign/exp/mantissa':>18s} {'largest':>11s} {'smallest normal':>16s} {'precision (eps)':>16s}")
for dt, (s, e, mant) in LAYOUT.items():
    fi = torch.finfo(dt)
    assert fi.bits == s + e + mant and fi.eps == 2.0 ** -mant       # the bit layout explains the numbers
    print(f"{str(dt)[6:]:9s} {f'{s} / {e} / {mant}':>18s} {fi.max:11.3g} {fi.tiny:16.3g} {fi.eps:16.3g}")

x_big, x_tiny = torch.tensor(70_000.0), torch.tensor(1e-8)
print("\n70,000 as fp16:", x_big.half().item(), "  as bf16:", x_big.bfloat16().item())
print("1e-8   as fp16:", x_tiny.half().item(), "      as bf16:", f"{x_tiny.bfloat16().item():.3g}")
w_bf16 = torch.tensor(1.0, dtype=torch.bfloat16)
print("1.0 + 0.001 in bf16:", (w_bf16 + 1e-3).item(), "  in fp32:", (torch.tensor(1.0) + 1e-3).item())
assert x_big.half().isinf() and x_tiny.half() == 0 and (w_bf16 + 1e-3).item() == 1.0

scale = 2.0 ** 16                                             # fp16 loss scaling: multiply up, divide back in fp32
g_scaled = (x_tiny * scale).half()
print(f"loss scaling: 1e-8 x 2^16 survives fp16 as {g_scaled.item():.3g}, unscaled back {g_scaled.float().item() / scale:.3g}")
assert g_scaled > 0 and abs(g_scaled.float().item() / scale - 1e-8) < 1e-10

- **fp16** has only 5 exponent bits: anything above 65,504 overflows to `inf`, and small gradients (1e-8)
  underflow to 0. Training in fp16 therefore needs **loss scaling**: multiply the loss by a large factor so the
  gradients land in range, then divide back in fp32 (`torch.amp.GradScaler` automates it).
- **bf16** keeps fp32's 8 exponent bits, so it has fp32's *range* and needs no loss scaling. It pays with
  precision: only 7 mantissa bits, so `1.0 + 0.001` rounds back to `1.0`. A small update added to a bf16 weight can
  vanish, which is why optimizers keep an **fp32 master copy** of the weights.
- Gradient descent tolerates imprecise arithmetic far better than overflow, which is why bf16 won.

`torch.autocast` runs each op in the lower precision where that is safe (matmuls) and keeps sensitive ones (like the
loss) in fp32. bf16 autocast works on CPU too:

In [ ]:
model = make_model()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
with torch.autocast(device_type="cpu", dtype=torch.bfloat16):    # on a GPU: device_type="cuda"
    logits = model(X_tr[:64])                                     # matmuls in bf16
    loss = F.cross_entropy(logits, y_tr[:64])                     # autocast keeps the loss in fp32
loss.backward()                                                   # backward outside the autocast block
opt.step()
print("logits:", logits.dtype, "| loss:", loss.dtype, "| weights:", model[0].weight.dtype,
      "| grads:", model[0].weight.grad.dtype)
assert logits.dtype == torch.bfloat16 and loss.dtype == torch.float32
assert model[0].weight.dtype == torch.float32 == model[0].weight.grad.dtype   # fp32 master weights

accs = {}
for label, enabled in [("fp32", False), ("bf16 autocast", True)]:
    model, gen = make_model(), torch.Generator().manual_seed(0)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for epoch in range(3):
        model.train()
        for xb, yb in batches(X_tr, y_tr, 64, gen):
            opt.zero_grad()
            with torch.autocast(device_type="cpu", dtype=torch.bfloat16, enabled=enabled):
                loss = F.cross_entropy(model(xb), yb)
            loss.backward(); opt.step()
    accs[label] = evaluate(model, X_va, y_va)[1]
print("val accuracy after 3 epochs:", {k: round(a, 3) for k, a in accs.items()})
assert abs(accs["fp32"] - accs["bf16 autocast"]) < 0.02

On a GPU with fp16, the loop adds a `GradScaler` (shown for reference, it needs CUDA):

```python
scaler = torch.amp.GradScaler("cuda")
with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss = loss_fn(model(xb), yb)
scaler.scale(loss).backward()        # scaled loss -> gradients stay in fp16 range
scaler.unscale_(opt)                 # back to true values so clipping sees real norms
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(opt)                     # skips the step if any gradient is inf/NaN
scaler.update()                      # adapts the scale factor
```

Memory per parameter when training with Adam (the arithmetic interviewers ask for):

| | fp32 training | bf16 mixed precision |
|---|---|---|
| weights used in forward/backward | 4 B | 2 B (bf16 copy) |
| fp32 master weights | (the same 4 B) | 4 B |
| gradients | 4 B | 2 B |
| Adam `m` and `v` | 8 B | 8 B (kept in fp32) |
| **total before activations** | **16 B** | **16 B** |

Mixed precision does **not** shrink the weights-plus-optimizer bill: a 7B-parameter full fine-tune needs about
7e9 × 16 B ≈ 112 GB either way. What it buys is half-size activations and much faster matmuls on tensor cores. The
state bill is cut by other tools: ZeRO/FSDP sharding, 8-bit optimizers, or LoRA (train few parameters).

## 11. Gradient checkpointing: trade compute for activation memory

Backward needs the activations saved during forward, and for long sequences they dominate memory. Checkpointing
stores only a segment's *input* and re-runs that segment's forward during backward. PyTorch lets us intercept every
tensor autograd saves for backward, so we can add up the bytes:

In [ ]:
from torch.utils.checkpoint import checkpoint

torch.manual_seed(0)
deep = nn.Sequential(*[nn.Sequential(nn.Linear(256, 256), nn.GELU()) for _ in range(8)])
weight_ptrs = {p.data_ptr() for p in deep.parameters()}
calls = [0]

def count_call(module, inputs, output):
    calls[0] += 1

for layer in deep:
    layer.register_forward_hook(count_call)
x = torch.randn(64, 256, requires_grad=True)

def run(use_checkpoint):
    calls[0], saved, x.grad = 0, [0], None
    def pack(t):                                   # runs for every tensor autograd keeps for backward
        if t.data_ptr() not in weight_ptrs:        # count activations, not weights (those are in memory anyway)
            saved[0] += t.numel() * t.element_size()
        return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        out = checkpoint(deep, x, use_reentrant=False) if use_checkpoint else deep(x)
    out.pow(2).mean().backward()
    return saved[0], calls[0], x.grad.clone()

(mem_plain, calls_plain, g_plain), (mem_ckpt, calls_ckpt, g_ckpt) = run(False), run(True)
print(f"activations kept for backward: {mem_plain / 1024:5.0f} KB plain   {mem_ckpt / 1024:5.0f} KB checkpointed")
print(f"layer forward calls per step:  {calls_plain:5d} plain   {calls_ckpt:5d} checkpointed (recomputed in backward)")
print("identical gradients:", torch.allclose(g_plain, g_ckpt))
assert torch.allclose(g_plain, g_ckpt) and mem_ckpt < mem_plain / 10 and calls_plain < calls_ckpt <= 2 * calls_plain

Same gradients, a sixteenth of the activation memory, and about one extra forward pass (roughly 20–35% more compute
per step). It is 15 layer calls rather than 16 because PyTorch stops recomputing as soon as it has every tensor the
backward pass needs.
In Hugging Face it is one call, `model.gradient_checkpointing_enable()`. The usual order of knobs when training hits
out-of-memory: smaller micro-batch + gradient accumulation, then checkpointing, then ZeRO/FSDP sharding, then LoRA
instead of full fine-tuning, then quantise the frozen base (QLoRA).

## Try it yourself

**1.** Warmup + cosine versus a constant learning rate, both with AdamW at an *aggressive* peak LR of 3e-2 for 10
epochs. Predict which ends with the lower validation loss, and which has the smoother curve.

In [ ]:
# Solution — try it yourself first, then run this
PEAK, EPOCHS = 3e-2, 10
n_steps = EPOCHS * math.ceil(len(X_tr) / 64)
curves = {}
for name in ["constant", "warmup + cosine"]:
    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=PEAK)
    sched = None if name == "constant" else LambdaLR(opt, lambda s: warmup_cosine(s, n_steps // 10, n_steps))
    curves[name] = fit(model, opt, epochs=EPOCHS, scheduler=sched)["val"]
    print(f"{name:16s} final val loss {curves[name][-1]:.3f}   spread over last 5 epochs {np.std(curves[name][-5:]):.3f}")
assert curves["warmup + cosine"][-1] < curves["constant"][-1]
assert np.std(curves["warmup + cosine"][-5:]) < np.std(curves["constant"][-5:])
print("(At a comfortable LR such as 1e-2 the difference mostly disappears on a problem this small.)")

**2.** Implement SGD with momentum by hand and check it against `torch.optim.SGD(momentum=0.9)` for 5 steps.
(PyTorch's form is `v = μ·v + g; w -= lr·v`.)

In [ ]:
# Solution — try it yourself first, then run this
torch.manual_seed(0)
w = nn.Parameter(torch.randn(4))
opt = torch.optim.SGD([w], lr=0.1, momentum=0.9)
w_hand, velocity = w.detach().clone(), torch.zeros(4)
for step in range(5):
    g = torch.randn(4)
    w.grad = g.clone()
    opt.step()
    velocity = 0.9 * velocity + g
    w_hand -= 0.1 * velocity
    assert torch.allclose(w.detach(), w_hand, atol=1e-6)
print("hand-written momentum SGD matches torch for 5 steps:", w_hand.tolist())

**3.** `clip_grad_value_` clamps each gradient element separately instead of rescaling the whole vector. Show that
it changes the gradient's *direction* while `clip_grad_norm_` does not.

In [ ]:
# Solution — try it yourself first, then run this
g = torch.tensor([0.1, 5.0, -20.0])
p_norm, p_value = nn.Parameter(torch.zeros(3)), nn.Parameter(torch.zeros(3))
p_norm.grad, p_value.grad = g.clone(), g.clone()
torch.nn.utils.clip_grad_norm_([p_norm], max_norm=1.0)
torch.nn.utils.clip_grad_value_([p_value], clip_value=1.0)
cosine = lambda a, b: (a @ b / (a.norm() * b.norm())).item()
print(f"clip_grad_norm_ : {p_norm.grad.tolist()}  cosine with original {cosine(p_norm.grad, g):.4f}")
print(f"clip_grad_value_: {p_value.grad.tolist()}  cosine with original {cosine(p_value.grad, g):.4f}")
assert cosine(p_norm.grad, g) > 0.9999 and cosine(p_value.grad, g) < 0.9

## Say it in an interview

- **The loop, in order:** `zero_grad` (gradients accumulate) → forward → loss on raw logits → `backward` → clip →
  `optimizer.step()` → `scheduler.step()`. Validate under `model.eval()` and `torch.no_grad()`.
- **Batch / step / epoch:** a batch is what one forward/backward sees, a step is one update, an epoch is one pass.
  Effective batch = micro-batch × accumulation steps × GPUs. LLM pretraining counts batches in tokens (millions per
  step) and rarely does more than about one epoch; fine-tuning usually runs 1–3 epochs.
- **Optimizers:** Adam keeps a running mean (`m`, direction) and a running mean square (`v`, per-weight scale) of
  the gradient, so every weight's step is normalised; the cost is 2 extra floats per parameter. AdamW decouples
  weight decay from that normalisation, and it is the default for transformers.
- **Learning rate first:** sweep it on a log grid. Too low barely moves, too high goes NaN. Scale it with batch
  size (linear scaling rule, with warmup). Gradient noise falls only as 1/√B.
- **The LLM recipe:** AdamW, linear warmup then cosine decay to about 10% of peak, `clip_grad_norm_(…, 1.0)`, bf16
  autocast, gradient accumulation to reach the target batch.
- **Mixed precision:** bf16 has fp32's exponent range, so no loss scaling (fp16 needs `GradScaler`); keep fp32
  master weights. Adam training costs ~16 bytes/parameter before activations with or without it; mixed precision
  saves activation memory and time.
- **Gradient checkpointing:** store segment inputs only, recompute in backward: much less activation memory for
  about one extra forward pass.
- **"My loss is NaN":** LR too high, no clipping, fp16 overflow (use bf16 or a GradScaler), NaN/inf in the data, or
  a hand-written `log(softmax)` instead of `F.cross_entropy`.
- **Traps:** calling the scheduler per epoch when it was built per step (or the reverse); forgetting `model.eval()`
  so dropout runs during validation; early stopping without restoring the best weights; comparing optimizers at one
  shared learning rate; expecting bit-identical GPU runs from a seed alone.

## Next

- [03 · Regularization, dropout and normalization](03_regularization_dropout_batchnorm.ipynb): closing the
  train/validation gap, weight decay done right, LayerNorm and RMSNorm.
- [04 · CNNs on digits](04_cnn_on_digits.ipynb): this loop, with a convolutional model.
- [Trainer API: fine-tune a small model](../10_huggingface/05_trainer_api_fine_tune_small_model.ipynb) (the same
  knobs as `TrainingArguments`) · [LoRA from scratch](../11_finetuning/03_lora_from_scratch.ipynb) ·
  [Quantization: int8, int4, nf4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)
- Theory: [dl_fundamentals course](../../dl_fundamentals/index.html) (chapters 6, 11 and 12) ·
  [interview bank](../../ai_interview_prep/index.html)